<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-classical-and-quantum-physics/blob/main/notebooks/lecture-08/exercise-03-koopman.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

<hr/>
# Deep Learning in Classical and Quantum Physics
2026/2027 <br>
**Author:** Marcin Płodzień <br>
**Exercise:** 03 <br>
**Related labs:** 035
<hr/>

# Exercise 03 — Koopman operator

The Koopman operator $\mathcal K_t$ advances observables, functions $g$ of the state, by composition with the flow map, $(\mathcal K_tg)(\mathbf x)=g(\varphi_t(\mathbf x))$. It is linear for every flow, and its generator is $\mathcal A g=\mathbf F\cdot\nabla g$ for the law of motion $\dot{\mathbf x}=\mathbf F(\mathbf x)$, so that $\mathcal K_t=e^{t\mathcal A}$. An eigenfunction $\xi$ with $\mathcal A\xi=\Lambda\xi$ evolves as $\xi(\mathbf x(t))=e^{\Lambda t}\xi(\mathbf x(0))$, and for data sampled with the interval $\Delta t$ the same eigenfunction is multiplied at every step by $\nu=e^{\Lambda\Delta t}$. A finite dictionary $\mathbf g=(g_1,\dots,g_M)$ is fitted by extended dynamic mode decomposition (EDMD),
$$
K=\Sigma_1(\Sigma_0+\beta I)^{-1},\qquad
\Sigma_0=\frac1{n_p}\sum_i\mathbf g(\mathbf x_i)\mathbf g(\mathbf x_i)^\top,\qquad
\Sigma_1=\frac1{n_p}\sum_i\mathbf g(\mathbf x_i')\mathbf g(\mathbf x_i)^\top,
$$
where $\mathbf x_i'$ is the state one step after $\mathbf x_i$, $n_p$ the number of pairs and $\beta\ge0$ a ridge penalty. With the state itself as the dictionary EDMD is dynamic mode decomposition (DMD), and with a window of past values of a measured signal it is delay DMD. The closure error $\mathbf g(\mathbf x')-K\mathbf g(\mathbf x)$ measures what the dictionary misses.

We work through four groups of exercises. In Section 1 we decide which polynomial dictionaries of a lifted map are closed. In Section 2 we show that DMD on a dictionary that does not close returns eigenvalues that depend on where the data lie. In Section 3 we construct the Koopman eigenfunctions of the damped and of the frictionless pendulum and compare them with an EDMD estimate. In Section 4 we fit delay DMD to a periodic and to a chaotic signal. The first code cell defines the EDMD fit, the closure error, the signal generators and the delay tools used throughout.

> **How to run this notebook in Google Colab**
> 1. Go to <https://colab.research.google.com> and upload the notebook with *File → Upload notebook*. The first code cell sets `JAX_PLATFORMS=cpu`, and the whole notebook runs in under a minute on a CPU.
> 2. `jax`, `sympy`, `scipy` and `matplotlib` are pre-installed in Colab.
> 3. Run the cells in order. Later cells use variables defined earlier.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"     # run JAX on the CPU (set before importing jax)

import json, time
from functools import partial
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import sympy as sp
from scipy.special import ellipk

import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)   # all maps, flows and fits in float64

# common style of the notebook figures
mpl.rcParams.update({"font.size": 14, "axes.labelsize": 15, "axes.titlesize": 15,
                     "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 12,
                     "figure.titlesize": 16, "lines.linewidth": 2.0,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
UJBlue, UJNavy, DSOrange, UJGray = "#005CA7", "#002B5C", "#D35A05", "#5A6270"

OUT = "figures"
os.makedirs(OUT, exist_ok=True)
def save(fig, name):
    # figures are saved as PDF and PNG
    for ext in ("pdf", "png"):
        fig.savefig(f"{OUT}/{name}.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.02)

nums = {}
t_start = time.time()

# ---------------------------------------------------------------- tools used in several exercises
def edmd(PX, PY, beta=0.0):
    """EDMD matrix K = Sigma_1 (Sigma_0 + beta I)^{-1} from dictionary values PX = g(x_i), PY = g(x_i'),
    both of shape (n_pairs, M)."""
    n = PX.shape[0]
    S0 = PX.T @ PX / n                      # Sigma_0, covariance of the dictionary with itself
    S1 = PY.T @ PX / n                      # Sigma_1, covariance with the dictionary one step later
    return np.linalg.solve((S0 + beta * np.eye(S0.shape[0])).T, S1.T).T

def closure_rms(K, PX, PY):
    """Root mean square of each component of the closure error g(x') - K g(x)."""
    return np.sqrt(np.mean((PY - PX @ K.T)**2, axis=0))

def rk4_step(f, u, h):
    k1 = f(u); k2 = f(u + 0.5*h*k1); k3 = f(u + 0.5*h*k2); k4 = f(u + h*k3)
    return u + h/6.0*(k1 + 2*k2 + 2*k3 + k4)

# Lorenz-63 (sigma=10, rho=28, beta=8/3), RK4 with h=0.01, sampled every 2 steps (dt=0.02),
# initial state (1,1,1), first 50 time units discarded
def lorenz_rhs(u, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = u
    return jnp.array([sigma*(y - x), x*(rho - z) - y, x*y - beta*z])

@partial(jax.jit, static_argnums=(0, 1, 2))
def lorenz_trajectory(n_samples, sub=2, n_discard=2500, h=0.01):
    def one_sample(u, _):
        u = jax.lax.fori_loop(0, sub, lambda i, v: rk4_step(lorenz_rhs, v, h), u)
        return u, u
    u, _ = jax.lax.scan(one_sample, jnp.array([1.0, 1.0, 1.0]), None, length=n_discard)
    _, traj = jax.lax.scan(one_sample, u, None, length=n_samples)
    return traj

# Mackey-Glass, dx/dt = beta x(t-tau)/(1+x(t-tau)^n) - gamma x with beta=0.2, gamma=0.1, n=10;
# RK4 with h=0.1, constant history 1.2, first 3000 fine steps discarded, every 10th point kept
def mackey_glass(tau, n_samples, h=0.1, x0=1.2, transient=3000, every=10, beta=0.2, gamma=0.1, n=10):
    lag = int(round(tau / h))
    f = lambda x, xd: beta * xd / (1.0 + xd**n) - gamma * x
    def step(buf, _):
        x, xd0, xd1 = buf[-1], buf[0], buf[1]
        xdh = 0.5 * (xd0 + xd1)
        k1 = f(x, xd0); k2 = f(x + 0.5*h*k1, xdh); k3 = f(x + 0.5*h*k2, xdh); k4 = f(x + h*k3, xd1)
        xn = x + h/6.0*(k1 + 2*k2 + 2*k3 + k4)
        return jnp.concatenate([buf[1:], xn[None]]), xn
    _, xs = jax.lax.scan(step, jnp.full(lag + 1, x0), None, length=transient + n_samples * every)
    x = np.concatenate([np.full(lag + 1, x0), np.asarray(xs)])
    return x[transient::every][:n_samples]

# protocol of the delay fits: N = 4000 samples, training block = first 3000, test block = last 1000;
# rescaling to [0, 1] with the minimum and maximum of the first 2500 samples, centring with the training mean
N_SAMPLES, N_TRAIN, N_FIT = 4000, 3000, 2500
def prepare(s):
    lo, hi = s[:N_FIT].min(), s[:N_FIT].max()
    s01 = (s - lo) / (hi - lo)
    return s01 - s01[:N_TRAIN].mean()

def hankel(s, L):
    """Delay vectors as columns, oldest sample first, shape (L, len(s) - L + 1)."""
    n = len(s) - L + 1
    return np.stack([s[j:j + n] for j in range(L)])

def delay_fit(s_train, L, beta=0.0):
    """Delay DMD on windows of L values. For beta = 0 the least-squares solution through the SVD, for beta > 0
    the ridge solution K = Sigma_1 (Sigma_0 + beta I)^{-1}."""
    U = hankel(s_train, L)
    if beta == 0.0:                              # least squares through the SVD, without forming Sigma_0
        return np.linalg.lstsq(U[:, :-1].T, U[:, 1:].T, rcond=None)[0].T
    return edmd(U[:, :-1].T, U[:, 1:].T, beta)

H_ROLL = 300
def rollout_scores(s, K, L):
    """Closed-loop forecasts u -> K u from every fifth test origin; normalised error and amplitude per lead."""
    last = np.arange(N_TRAIN + L - 1, N_SAMPLES - H_ROLL, 5)          # newest sample of each initial window
    U = hankel(s, L)
    u = U[:, last - L + 1]                                             # (L, n_origins)
    preds = []
    for _ in range(H_ROLL):
        u = K @ u
        preds.append(u[-1])
    preds = np.array(preds).T                                          # (n_origins, H_ROLL)
    truth = np.stack([s[i + 1:i + 1 + H_ROLL] for i in last])
    sd = s[N_TRAIN:].std()
    nrmse = np.sqrt(np.mean((preds - truth)**2, axis=0)) / sd
    amp = np.sqrt(np.mean(preds**2, axis=0)) / sd
    return nrmse, amp, len(last)
print("jax", jax.__version__, "| sympy", sp.__version__)

<hr/>
## 1. Closed dictionaries of a lifted map

The map
$$
x_{k+1}=a\,x_k,\qquad y_{k+1}=b\,y_k+c\,x_k^2
$$
with real parameters $a$, $b$, $c$ becomes linear on the three observables $(x,y,x^2)$, with the matrix
$$
K_3=\begin{pmatrix}a&0&0\\0&b&c\\0&0&a^2\end{pmatrix}
$$
and the eigenfunctions $\xi_a=x$, $\xi_{a^2}=x^2$ and $\xi_b=y+\kappa x^2$ with $\kappa=c/(b-a^2)$. Adding the single observable $y^2$ to this dictionary leaves a closure error in the new component, since $y^2$ at the next step contains terms outside the span. We use $a=0.9$, $b=0.5$ and $c=1$ for all numerical checks.

### Exercise 1.1. A closed dictionary that contains $y^2$

Compute the images of $y^2$, $x^2y$ and $x^4$ under one step of the map and show that the six observables $\mathbf g=(x,\,y,\,x^2,\,y^2,\,x^2y,\,x^4)$ span an invariant subspace. Write the $6\times6$ matrix $K_6$ with $\mathbf g(\mathbf x_{k+1})=K_6\,\mathbf g(\mathbf x_k)$ and check $\mathbf g\circ\varphi-K_6\mathbf g=0$ with `sympy`. Find the eigenvalues of $K_6$ and express them through $a$ and $b$. Show that $\xi_b^2$ and $x^2\xi_b$ are eigenfunctions and identify their eigenvalues.

Then verify the closure numerically. Fit EDMD with $\beta=0$ on 500 states drawn uniformly from $[-1,1]^2$ for the dictionaries $(x,y,x^2,y^2)$ and $\mathbf g$, and compute the root-mean-square (RMS) closure error of every component on 2000 fresh states. Compare the fitted matrix with $K_6$, and compare the 50-step forecast of $y^2$ by the matrix power $K_6^n$ with direct iteration of the map from 200 random states.

In [ ]:
# your code: symbolic images of y^2, x^2 y, x^4, the matrix K6 and its eigenvalues; EDMD closure errors
# for (x, y, x^2, y^2) and the six-function dictionary; 50-step forecast of y^2

### Exercise 1.2. Which polynomial dictionaries close

Show that one step maps the monomial $x^my^n$ to a combination of the monomials $x^{m+2j}y^{n-j}$ with $j=0,\dots,n$. Assign the weight 1 to $x$ and the weight 2 to $y$ and show that the weighted degree $m+2n$ is preserved. State the rule that a closed dictionary of monomials must satisfy, find the smallest closed dictionary that contains $y^3$, and confirm its closure with EDMD on the states of Exercise 1.1. Compare with the RMS closure error of the dictionary $(y^3)$ alone.

Then take the one-dimensional map $x_{k+1}=a\,x_k+e\,x_k^2$ with $a=0.9$ and $e=0.1$. Fit EDMD on the powers $(x,x^2,\dots,x^D)$ for $D=1,\dots,8$ with 500 states uniform in $[-1,1]$, and report the RMS closure error of $x$ and of the highest power $x^D$ on 2000 fresh states.

In [ ]:
# your code: weighted degree of the images; closure of (y^3, x^2 y^2, x^4 y, x^6) and of (y^3);
# powers x, ..., x^D for the map a x + e x^2

<hr/>
## 2. DMD and the Koopman operator

DMD fits a linear map to the state itself, which for the map of Section 1 is the dictionary $(x,y)$. This dictionary does not close, since $y$ at the next step contains $c\,x^2$. For a dictionary that does not close, the least-squares fit is the projection of $\mathcal K$ onto the span of the dictionary in the inner product defined by the data, $\langle f,h\rangle=\frac1{n_p}\sum_if(\mathbf x_i)h(\mathbf x_i)$, and its result depends on where the data lie. For a closed dictionary the projection is exact and the fitted matrix is the same for any data that span the dictionary.

### Exercise 2.1. DMD eigenvalues depend on the data

(a) Let $x$ and $y$ be independent and uniform on $[-1,1]$. Compute $\Sigma_0$ and $\Sigma_1$ of the dictionary $(x,y)$ in the limit of infinitely many pairs, show that the fitted matrix is $\mathrm{diag}(a,b)$, and estimate how the fitted eigenvalue near $b$ scatters for $N$ pairs. Check the scatter over 200 independent samples for $N=125$, 500, 2000 and 8000.

(b) Repeat the limit calculation for $y$ uniform on $[0,1]$ and check it with a sample of $2\times10^5$ pairs.

(c) Fit DMD on the 500 states of 20 trajectories of 25 steps each, started from random points of $[-1,1]^2$, and report the eigenvalues.

(d) Fit EDMD with the closed dictionary $(x,y,x^2)$ for the three samplings and compare with $K_3$. Explain in two sentences what the comparison says about the relation between DMD eigenvalues and Koopman eigenvalues.

In [ ]:
# your code: scatter of the DMD eigenvalue over 200 samples; DMD for y in [0, 1] and along trajectories;
# EDMD with (x, y, x^2) for the three samplings

<hr/>
## 3. Koopman eigenfunctions of the pendulum

The damped pendulum
$$
\dot\theta=\omega,\qquad \dot\omega=-\omega_p^2\sin\theta-\gamma\,\omega
$$
has the angle $\theta$, the angular velocity $\omega$, the small-amplitude frequency $\omega_p$ and the damping rate $\gamma$. We use $\omega_p=1$ and $\gamma=0.2$. The hanging position $(0,0)$ is a stable spiral and the upright position $(\pi,0)$ a saddle. The mechanical energy per unit moment of inertia is $E=\tfrac12\omega^2+\omega_p^2(1-\cos\theta)$. The generator acts as $\mathcal Ag=\omega\,\partial_\theta g-(\omega_p^2\sin\theta+\gamma\omega)\,\partial_\omega g$.

### Exercise 3.1. Eigenvalues at the stable fixed point and the energy

Compute the Jacobian of the vector field at $(0,0)$ and its eigenvalues $\Lambda_\pm$. For each eigenvalue find the left eigenvector $\mathbf l=(l_1,1)$ with $\mathbf lJ=\Lambda\mathbf l$ and form the linear function $\xi^{\rm lin}(\theta,\omega)=l_1\theta+\omega$. Compute $\mathcal A\xi^{\rm lin}-\Lambda\xi^{\rm lin}$ with `sympy` and explain why $\xi^{\rm lin}$ is an eigenfunction only near the fixed point. Compute $\mathcal AE$ with and without friction and state what follows for the Koopman operator of the frictionless pendulum. Evaluate $\Lambda_+$ for $\omega_p=1$ and $\gamma=0.2$.

In [ ]:
# your code: Jacobian, eigenvalues, left eigenvectors and the defect of xi_lin with sympy; A E with and without friction

### Exercise 3.2. The principal eigenfunction away from the fixed point

The flow carries every state below the energy $2\omega_p^2$ of the upright position to the hanging position, where $\xi^{\rm lin}$ is accurate. This suggests the construction
$$
\xi(\mathbf x)=\lim_{T\to\infty}e^{-\Lambda_+T}\,\xi^{\rm lin}\big(\varphi_T(\mathbf x)\big).
$$
Show that a function defined in this way obeys $\xi(\varphi_t(\mathbf x))=e^{\Lambda_+t}\xi(\mathbf x)$. Evaluate the limit numerically with RK4 steps of 0.01 for 400 random states with $E<1.9$, check its convergence in $T$ and the eigenvalue relation at $t=5$, and measure how far $\xi$ differs from $\xi^{\rm lin}$ within the radius $|\mathbf x|\le r$ for $r=0.25$, 0.5, 1 and 1.5. Plot the level curves of $|\xi|$ and the phase $\arg\xi$ in the phase plane.

In [ ]:
# your code: principal eigenfunction as a limit along the RK4 flow; convergence, eigenvalue check, deviation from
# the linear part; contour plot of |xi| and colour map of arg xi

### Exercise 3.3. EDMD for the pendulum

Fit EDMD with all monomials $1,\theta,\omega,\theta^2,\theta\omega,\dots,\omega^7$ of total degree up to 7 (36 functions) on 4000 pairs of states one time step $\Delta t=0.1$ apart, the first state of each pair uniform in $[-1,1]^2$, with $\beta=10^{-8}$. Convert the eigenvalues $\nu$ of the fitted matrix to $\Lambda=\ln\nu/\Delta t$.

(a) For $\gamma=0.2$ compare the EDMD values with $\Lambda_+$, $2\Lambda_+$ and $\Lambda_++\Lambda_-=-\gamma$, and compare the EDMD eigenfunction belonging to $\Lambda_+$, the left eigenvector of the fitted matrix contracted with the dictionary, with $\xi$ of Exercise 3.2 on 500 test states, after fixing the free complex factor of the eigenfunction by least squares.

(b) For $\gamma=0$ find the eigenvalues near zero and check whether $E$ is reproduced by a constant plus a multiple of one of their eigenfunctions. The frequency of a frictionless swing of energy $E<2\omega_p^2$ is $\Omega(E)=\pi\omega_p/(2\mathsf K(k))$ with $k=\sin(\theta_{\max}/2)$, $E=\omega_p^2(1-\cos\theta_{\max})$ and the complete elliptic integral of the first kind $\mathsf K$ (`scipy.special.ellipk` takes $k^2$). Compute the range of $\Omega$ for the energies of the sampled square, check $\Omega$ against the period of an RK4 trajectory at the largest energy, and compare with the oscillation frequencies $|\mathrm{Im}\,\Lambda|$ that EDMD returns.

In [ ]:
# your code: EDMD with monomials of degree <= 7 for gamma = 0.2 and gamma = 0; comparison with Lambda_+, the eigenfunction
# xi and the energy; Omega(E) from the elliptic integral and from an RK4 period

<hr/>
## 4. Delay DMD of measured signals

A record offers only the values $s_k=h(\mathbf x_k)$ of one observable, and the window $\mathbf u_k=(s_k,s_{k-1},\dots,s_{k-L+1})$ is a dictionary generated by the dynamics from $h$. A linear map fitted from $\mathbf u_k$ to $\mathbf u_{k+1}$ is delay DMD, and iterating it gives a recursive forecast. Its first row is the least-squares autoregressive model of order $L$. Both signals below have $N=4000$ samples, are rescaled to $[0,1]$ with the minimum and maximum of the first 2500 samples and centred with the mean of the first 3000, the training block. The forecasts start from every fifth origin of the test block, the last 1000 samples, for which the window and 300 forecast steps lie inside the block. The normalised root-mean-square error $\mathrm{NRMSE}(\tau)$ is the root-mean-square difference between forecast and truth at the lead $\tau$ over the origins, divided by the standard deviation of the test block, and the amplitude $A(\tau)$ is the root mean square of the forecast itself divided by the same standard deviation.

### Exercise 4.1. Window length for a limit cycle

Fit delay DMD to the Mackey–Glass signal with the delay $\tau_{\rm MG}=15$ for the windows $L=20$, 40, 60, 80, 85, 88, 90 and 100, using the least-squares solution through the singular value decomposition. Report the largest NRMSE over 300 steps and the spectral radius of each fitted matrix. Measure the period of the limit cycle from the maxima of the signal, which repeat with three maxima per period, and relate the smallest window that follows the cycle to the period.

In [ ]:
# your code: period of the Mackey-Glass cycle; delay DMD for the listed windows; largest NRMSE and spectral radius

### Exercise 4.2. Ill-conditioned fits and the ridge penalty

Fit delay DMD with $L=30$ to the Lorenz $x$ signal (sampling interval $\Delta t=0.02$) without a penalty in three ways, with `numpy.linalg.lstsq`, with `numpy.linalg.pinv` at the relative cutoff $10^{-12}$, and with a pseudoinverse through the singular value decomposition in JAX at the same cutoff. Report the spectral radius, the forecast amplitude $A(300)$ and the largest difference between the matrices, together with the condition number of the matrix of delay vectors. Then fit the ridge solution $K=\Sigma_1(\Sigma_0+\beta I)^{-1}$ for $\beta=10^{-12},10^{-10},\dots,10^{-3}$ and report the spectral radius and the NRMSE at the lead of 14 samples and at one Lyapunov time, 55 samples ($T_\lambda=1/\lambda_1=1.11$ time units with the largest Lyapunov exponent $\lambda_1=0.902$). Compare with the fit at $L=20$.

In [ ]:
# your code: three unpenalised solvers at L = 30; ridge fits for the listed beta; spectral radius and NRMSE

<hr/>
### References


- B. O. Koopman, Hamiltonian systems and transformation in Hilbert space, Proc. Natl. Acad. Sci. USA **17**, 315 (1931), [doi:10.1073/pnas.17.5.315](https://doi.org/10.1073/pnas.17.5.315).
- P. J. Schmid, Dynamic mode decomposition of numerical and experimental data, J. Fluid Mech. **656**, 5 (2010), [doi:10.1017/S0022112010001217](https://doi.org/10.1017/S0022112010001217).
- M. O. Williams, I. G. Kevrekidis and C. W. Rowley, A data-driven approximation of the Koopman operator. Extending dynamic mode decomposition, J. Nonlinear Sci. **25**, 1307 (2015), [doi:10.1007/s00332-015-9258-5](https://doi.org/10.1007/s00332-015-9258-5).
- I. Mezić, Spectral properties of dynamical systems, model reduction and decompositions, Nonlinear Dyn. **41**, 309 (2005), [doi:10.1007/s11071-005-2824-x](https://doi.org/10.1007/s11071-005-2824-x).
- A. Mauroy, I. Mezić and J. Moehlis, Isostables, isochrons, and Koopman spectrum for the action–angle representation of stable fixed point dynamics, Physica D **261**, 19 (2013), [doi:10.1016/j.physd.2013.06.004](https://doi.org/10.1016/j.physd.2013.06.004).
- S. L. Brunton, M. Budišić, E. Kaiser and J. N. Kutz, Modern Koopman theory for dynamical systems, SIAM Rev. **64**, 229 (2022), [doi:10.1137/21M1401243](https://doi.org/10.1137/21M1401243).